# TradeSense · Component 03 · notebook 59: explanations that respect the data (protocol E4 v1; CPU, minutes)

**Question.** Notebooks 51 and 55 reported the XGBoost explanation checks as FAILED. Protocol E4 v1 (written before any number) asks the same question on both sides of the comparison: what happens to the score when a feature is replaced by what the market normally looks like (50 background bars from CAL-3). XGBoost gets exact interventional Shapley values, the Transformer a background what-if. The results of notebooks 51 and 55 are not edited.

**Order of work.** (1) protocol hash, (2) frozen models and TEST events, (3) background written, (4) unit tests, (5) results. Nothing is retrained.

**Protocol text hashed by this notebook:**

```
# Protocol E4 v1 (go-ahead given by the user 2026-10-18; frozen when notebook 59 hashes this exact text) - explanations that respect the data: background-marginalised (interventional) attribution for XGBoost, background what-if for the Transformer (novelty 3, second follow-up to H-C5 / H-C9 of protocol E2 and H-E1x..H-E3x of protocol E3 v1)
Written BEFORE any number of this analysis exists. Nothing frozen changes (models, 6 features, normalisation, fusion stacker, thresholds). CPU, no training. Notebook 51 and notebook 55 results stay as reported and are not edited. Thresholds below are the SAME as in E3 v1; they are not lowered.

## Why the XGBoost explanations failed in E3 (diagnosis, no new number)
TreeSHAP (path-dependent) credits a feature by following the tree paths the training data took, so it explains the model on the data distribution; the deletion test of E3 sets the last 64 bars of a feature to a constant and recomputes 53 derived features, a state that never occurs in the data. The two answer different questions, and in notebook 55 even the "what if" column itself missed the injected signature (0.61 vs chance 0.62) while the attribution found it (0.95). A fair comparison needs both sides to ask the same question: what happens to the score if the information of a feature is replaced by what the market normally looks like. That is the interventional (background-marginalised) definition of Shapley values.

## Part A - the explanation (declared now)
- Background: K = 50 bars per symbol drawn once with seed 11 from CAL-3 (first 31 days) bars that are at least 130 bars away from every CAL-3 event and have finite scores. For the Transformer the same bars give K = 20 background windows (the first 20 of the 50). The background is written to the file explain_background_v1.npz BEFORE any TEST event is read and never changed.
- XGBoost value function: for a set S of the six Layer-1 features, v(S) = mean over the K background rows of the model margin of the event's 53-feature row in which the derived columns of every feature NOT in S (the columns whose base name is that feature) are replaced by the background row's values; the cross-feature columns "x" stay at the event's values. Exact Shapley values phi_j (six features, all 64 coalitions) with efficiency v(all) - v(empty) = sum phi_j. Group attribution = sum of phi over the group (groups of E3 v1: price, flow, size_volume).
- Marginal drops: single-feature drop d_j = v(all) - v(all minus j); group drop D_g = v(all) - v(all minus the two features of g).
- Transformer what-if: for feature j, replace the last 64 bars of column j of the event window by the same columns of each background window, average the output over the 20 windows; drop = output(original) - that mean. Attribution stays gradient x input (E3 v1).
- Events: the same model-active TEST events as notebook 55 (Sep and Oct 2024, t0 >= 44,640; peak bar above the model's own alert line; XGBoost 431, Transformer 412 expected). This is a NEW analysis of the same events with a new explanation method; it is not an independent sample, which is a stated limit.

## Part B - hypotheses
XGBoost (interventional group Shapley):
- H-E1y: the top group by attribution equals the top group by group drop D_g in >= 0.60 of events, 95% lower bound (event bootstrap, 2,000 draws, seed 0) > 1/3.
- H-E2y: the top feature by phi is within the two largest single-feature drops d_j in >= 0.65 of events, lower bound > 1/3.
- H-E3y: the top feature by phi is in the injected event signature (SIGN of notebook 51) in >= (signature chance + 0.15) of events and the bootstrap lower bound of (hit - chance) > 0.
Transformer:
- H-E4y: the top feature of the background what-if drop is in the injected signature in >= chance + 0.15 and not worse than the gradient x input attribution by more than 0.03 (paired bootstrap lower bound of the difference > -0.03).
Exactness:
- H-E5y: efficiency |sum phi - (v(all) - v(empty))| <= 1e-6 for every event; v(all) equals the booster margin of the event row (<= 1e-3); no background bar lies within 130 bars of a CAL-3 event; a toy linear model reproduces its closed-form Shapley values (T-tests).
Descriptive (no decision): the path-dependent TreeSHAP group hit against the same marginal drops (for comparison with E3 v1), the same for random attribution, per-family tables, Spearman between phi and the drops.

## Product rule (declared now)
If H-E1y and H-E3y hold, the dashboard computes this attribution for the explained XGBoost minute (64 coalitions x 50 background rows, about 3,200 rows, on demand) from the background file and labels the XGBoost group reason SUPPORTED with the measured test numbers; otherwise the XGBoost reasons stay INDICATIVE. If H-E4y holds the Transformer what-if column may be shown as SUPPORTED, otherwise it stays hidden.

## Known weaknesses, stated in advance
Cross-feature columns are kept at the event's values, so a replaced feature can leave inconsistent cross terms; K = 50 is a fixed choice (not tuned); explanations describe the model's score, not the cause of a real market event; labels and signatures are synthetic. A failed hypothesis is reported as failed and nothing is retuned.

```


In [1]:
import os, sys, json, glob, time, hashlib, warnings, datetime
from pathlib import Path
import numpy as np, pandas as pd
from scipy.stats import spearmanr
warnings.filterwarnings('ignore')
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not in Colab -> using TRADESENSE_BASE.')
import torch, xgboost as xgb
BASE = Path(os.environ.get('TRADESENSE_BASE', '/content/drive/MyDrive/research datasets'))
L3 = BASE/'layer3_v3'; OUT = BASE/'models_b2_v1'; SYMBOLS = ['BTCUSDT', 'ETHUSDT']
MOCK = os.environ.get('TS_MOCK') == '1'; NBOOT = 300 if MOCK else 3000
LINES = []
def say(s=''): print(s); LINES.append(str(s))
def sha_file(p, n=16): return hashlib.sha256(Path(p).read_bytes()).hexdigest()[:n]
hf = L3/'protocol_bar_models_v2.sha256'; pf = L3/'protocol_bar_models_v2.md'
assert hf.exists() and pf.exists(), 'protocol B2 missing'
ph = hashlib.sha256(pf.read_text().encode()).hexdigest(); assert ph == hf.read_text().split()[0], 'frozen protocol B2 was changed'
PROTO_E4 = r"""# Protocol E4 v1 (go-ahead given by the user 2026-10-18; frozen when notebook 59 hashes this exact text) - explanations that respect the data: background-marginalised (interventional) attribution for XGBoost, background what-if for the Transformer (novelty 3, second follow-up to H-C5 / H-C9 of protocol E2 and H-E1x..H-E3x of protocol E3 v1)
Written BEFORE any number of this analysis exists. Nothing frozen changes (models, 6 features, normalisation, fusion stacker, thresholds). CPU, no training. Notebook 51 and notebook 55 results stay as reported and are not edited. Thresholds below are the SAME as in E3 v1; they are not lowered.

## Why the XGBoost explanations failed in E3 (diagnosis, no new number)
TreeSHAP (path-dependent) credits a feature by following the tree paths the training data took, so it explains the model on the data distribution; the deletion test of E3 sets the last 64 bars of a feature to a constant and recomputes 53 derived features, a state that never occurs in the data. The two answer different questions, and in notebook 55 even the "what if" column itself missed the injected signature (0.61 vs chance 0.62) while the attribution found it (0.95). A fair comparison needs both sides to ask the same question: what happens to the score if the information of a feature is replaced by what the market normally looks like. That is the interventional (background-marginalised) definition of Shapley values.

## Part A - the explanation (declared now)
- Background: K = 50 bars per symbol drawn once with seed 11 from CAL-3 (first 31 days) bars that are at least 130 bars away from every CAL-3 event and have finite scores. For the Transformer the same bars give K = 20 background windows (the first 20 of the 50). The background is written to the file explain_background_v1.npz BEFORE any TEST event is read and never changed.
- XGBoost value function: for a set S of the six Layer-1 features, v(S) = mean over the K background rows of the model margin of the event's 53-feature row in which the derived columns of every feature NOT in S (the columns whose base name is that feature) are replaced by the background row's values; the cross-feature columns "x" stay at the event's values. Exact Shapley values phi_j (six features, all 64 coalitions) with efficiency v(all) - v(empty) = sum phi_j. Group attribution = sum of phi over the group (groups of E3 v1: price, flow, size_volume).
- Marginal drops: single-feature drop d_j = v(all) - v(all minus j); group drop D_g = v(all) - v(all minus the two features of g).
- Transformer what-if: for feature j, replace the last 64 bars of column j of the event window by the same columns of each background window, average the output over the 20 windows; drop = output(original) - that mean. Attribution stays gradient x input (E3 v1).
- Events: the same model-active TEST events as notebook 55 (Sep and Oct 2024, t0 >= 44,640; peak bar above the model's own alert line; XGBoost 431, Transformer 412 expected). This is a NEW analysis of the same events with a new explanation method; it is not an independent sample, which is a stated limit.

## Part B - hypotheses
XGBoost (interventional group Shapley):
- H-E1y: the top group by attribution equals the top group by group drop D_g in >= 0.60 of events, 95% lower bound (event bootstrap, 2,000 draws, seed 0) > 1/3.
- H-E2y: the top feature by phi is within the two largest single-feature drops d_j in >= 0.65 of events, lower bound > 1/3.
- H-E3y: the top feature by phi is in the injected event signature (SIGN of notebook 51) in >= (signature chance + 0.15) of events and the bootstrap lower bound of (hit - chance) > 0.
Transformer:
- H-E4y: the top feature of the background what-if drop is in the injected signature in >= chance + 0.15 and not worse than the gradient x input attribution by more than 0.03 (paired bootstrap lower bound of the difference > -0.03).
Exactness:
- H-E5y: efficiency |sum phi - (v(all) - v(empty))| <= 1e-6 for every event; v(all) equals the booster margin of the event row (<= 1e-3); no background bar lies within 130 bars of a CAL-3 event; a toy linear model reproduces its closed-form Shapley values (T-tests).
Descriptive (no decision): the path-dependent TreeSHAP group hit against the same marginal drops (for comparison with E3 v1), the same for random attribution, per-family tables, Spearman between phi and the drops.

## Product rule (declared now)
If H-E1y and H-E3y hold, the dashboard computes this attribution for the explained XGBoost minute (64 coalitions x 50 background rows, about 3,200 rows, on demand) from the background file and labels the XGBoost group reason SUPPORTED with the measured test numbers; otherwise the XGBoost reasons stay INDICATIVE. If H-E4y holds the Transformer what-if column may be shown as SUPPORTED, otherwise it stays hidden.

## Known weaknesses, stated in advance
Cross-feature columns are kept at the event's values, so a replaced feature can leave inconsistent cross terms; K = 50 is a fixed choice (not tuned); explanations describe the model's score, not the cause of a real market event; labels and signatures are synthetic. A failed hypothesis is reported as failed and nothing is retuned.
"""
PHE = hashlib.sha256(PROTO_E4.encode()).hexdigest(); pp = OUT/'protocol_e4_v1.md'
if pp.exists(): assert pp.read_text() == PROTO_E4, 'protocol E3 exists and differs from this notebook'
else: pp.write_text(PROTO_E4)
(OUT/'protocol_e4_v1.sha256').write_text(PHE + '  protocol_e4_v1.md\n'); assert (OUT/'declared_explain_v2.json').exists(), 'run notebook 51 first'
for f in ['declared_final_set_v1.json', 'declared_ae_v2.json', 'declared_explain_v1.json', 'test_b_results_v1.json', 'b2_lib.py', 'c03_infer.py']: assert (OUT/f).exists(), f'{f} missing'
sys.path.insert(0, str(OUT)); import b2_lib as B; import c03_infer as c
torch.set_num_threads(2)
say('protocol E4 v1 sha256 ' + PHE[:16] + '; protocol B2 intact ' + ph[:16] + '; library ' + sha_file(OUT/'c03_infer.py') + '; xgboost ' + xgb.__version__ + ', torch ' + torch.__version__ + ('  [MOCK RUN]' if MOCK else ''))

Mounted at /content/drive
protocol E4 v1 sha256 7a0a2d108008e69b; protocol B2 intact 443f99d5faf5929f; library 3b5d9df23521adf9; xgboost 3.4.1, torch 2.11.0+cpu


## 1 · Frozen models, TEST features and events

In [2]:
AOFULL = 132480; AOLEN = AOFULL; TEST_LO = 31 * 1440; TAGM = dict(zip(c.MODEL_NAMES, ['AE', 'XGB', 'TR'])); WARM = B.WARM; DELLEN = 64; PRE = 130
T32 = json.load(open(OUT/'test_b_results_v1.json')); DF = json.load(open(OUT/'declared_final_set_v1.json')); MODELS = c.MODEL_NAMES; MODES = {m: DF['models'][m]['normalisation'] for m in MODELS}; FILES = {MODELS[0]: 'ae_ctx_agg', MODELS[1]: 'xgb_v2', MODELS[2]: 'transformer_v2'}
SIGN = {1: ['log_returns', 'trade_flow_imbalance', 'hl_spread_ratio', 'volume_acceleration', 'large_trade_count'], 2: ['log_returns', 'trade_flow_imbalance', 'hl_spread_ratio', 'volume_acceleration', 'large_trade_count'],
        3: ['volume_acceleration', 'large_trade_count', 'rush_order_density'], 4: ['trade_flow_imbalance', 'log_returns'], 5: ['rush_order_density', 'hl_spread_ratio', 'trade_flow_imbalance'],
        6: ['hl_spread_ratio', 'log_returns', 'large_trade_count'], 7: ['log_returns', 'volume_acceleration', 'large_trade_count', 'trade_flow_imbalance', 'hl_spread_ratio'], 8: ['trade_flow_imbalance', 'log_returns', 'large_trade_count', 'volume_acceleration']}
FEATS = B.FEATS; BASEOF = [n.split('__')[0] for n in c.FA_NAMES]; GROUPS = FEATS + ['x']; GIDX = {g_: [i for i, n in enumerate(BASEOF) if n == g_] for g_ in GROUPS}; assert sum(len(v) for v in GIDX.values()) == 53
def find1(pat):
    cc = [x for x in glob.glob(str(BASE/'**'/pat), recursive=True) if 'backup' not in x.lower()]; assert cc, pat; return sorted(cc)[0]
N0 = {}; AEC = {}; BUN = {}; FR = {}; EV = {}; SAV = {}; THR = {}
for s in SYMBOLS:
    a = pd.read_parquet(find1(f'{s.lower()}_l1_feature_matrix_90d.parquet')); a = a.set_index('datetime') if 'datetime' in a.columns else a
    N0[s] = B.fit_n0(a[FEATS].reset_index(drop=True).iloc[15:90720]); AEC[s] = np.load(OUT/'scores'/f'ae_ctx_{s}_N1.npz'); BUN[s] = c.load_bundle_from_training(OUT, s, DF, AEC[s]['scale'], N0[s], 'cpu')
    f = pd.read_parquet(L3/f'l3v3_{s}_TEST.parquet'); FR[s] = f[f.replica == 0].reset_index(drop=True).iloc[:AOLEN][FEATS]; e = pd.read_csv(L3/f'l3v3_{s}_TEST_events.csv'); EV[s] = e[e.replica == 0].reset_index(drop=True); EV[s] = EV[s][EV[s].t0 >= TEST_LO].reset_index(drop=True)
    assert sha_file(OUT/'scores'/f'test_b_{s}.npz') == T32['test_score_files_sha16'][s], 'saved TEST scores differ from what notebook 32 recorded'; z = np.load(OUT/'scores'/f'test_b_{s}.npz')
    for m in MODELS[1:]:
        SAV[(m, s)] = c.nanw(z[TAGM[m] + '__r0'][:AOLEN]); THR[(m, s)] = c.rolling_threshold(c.nanw(z[TAGM[m] + '__GEN'][:AOLEN]), DF['models'][m]['q_T1_by_budget']['2'][s])
Z = {(m, s): np.asarray(B.normalise(FR[s], MODES[m], N0[s]), 'float64') for s in SYMBOLS for m in MODELS[1:]}
say('TEST events per symbol (Sep and Oct only) loaded; frozen models loaded; declared XGBoost ' + DF['models'][MODELS[1]]['configuration'] + ' (' + MODES[MODELS[1]] + '), Transformer ' + DF['models'][MODELS[2]]['configuration'] + ' (' + MODES[MODELS[2]] + '); CAL-3 events per symbol ' + str({s: len(EV[s]) for s in SYMBOLS}))

TEST events per symbol (Sep and Oct only) loaded; frozen models loaded; declared XGBoost A|pooled|d3 (N0), Transformer W16_d32_L1 (N1); CAL-3 events per symbol {'BTCUSDT': 300, 'ETHUSDT': 300}


## 2 · Functions (from notebook 51)

In [3]:
def xgb_x(s, z, b):
    """53 features of the last row of a short normalised slice ending at bar b"""
    return c.build_feats(z[b - 120:b + 1])[0][-1:]
def xgb_margin(s, X):
    dm = xgb.DMatrix(X, feature_names=c.FA_NAMES); return float(np.mean([b_.predict(dm, output_margin=True, iteration_range=(0, bi + 1)) for b_, bi in BUN[s].xgb], axis=0)[0])
def xgb_shap(s, X):
    dm = xgb.DMatrix(X, feature_names=c.FA_NAMES); con = np.mean([b_.predict(dm, pred_contribs=True, iteration_range=(0, bi + 1)) for b_, bi in BUN[s].xgb], axis=0)[0]; return con[:-1], con[-1]
def shap6(con): return np.array([con[GIDX[g_]].sum() for g_ in FEATS])
def delete_xgb(s, z, b, j):
    zz = z[b - 120:b + 1].copy(); zz[-DELLEN:, j] = 0.0; return c.build_feats(zz)[0][-1:]
def tr_out(s, win):
    with torch.no_grad(): x = torch.tensor(np.asarray(win, 'float32')[None]); return float(np.mean([n_(x).float().item() for n_ in BUN[s].tr_nets]))
def tr_gradxinput(s, win):
    """mean over the declared nets of d output / d input, times input; returns (W, 6) and the mean gradient"""
    x = torch.tensor(np.asarray(win, 'float32')[None]); x.requires_grad_(True); y = 0.0
    for n_ in BUN[s].tr_nets: y = y + n_(x).sum() / len(BUN[s].tr_nets)
    y.backward(); g = x.grad[0].numpy().astype('float64'); return g * np.asarray(win, 'float64'), g
def delete_tr(win, j):
    w2 = np.asarray(win, 'float32').copy(); w2[-DELLEN:, j] = 0.0; return w2
def explain_tgnn(*args, **kwargs):
    """Interface for the TGNN explanation. Nothing is produced until the TGNN is integrated and tested."""
    return {'available': False, 'reason': 'TGNN is not integrated yet; no explanation is produced or claimed', 'method': None, 'feature_attribution': None, 'time_attribution': None, 'faithfulness_tested': False}

## 3 · Model-active events

In [4]:
def peak_bar(s, m, ev):
    sc = SAV[(m, s)]; ii = np.arange(ev.t0, ev.t0 + ev.dur); v = sc[ii]; v = np.where(np.isfinite(v), v, -np.inf); return int(ii[int(np.argmax(v))]) if np.isfinite(v).any() and v.max() > -np.inf else None
XE = []; TE = []; excl = {'XGBoost v2': 0, 'Transformer v2': 0}; W_ = BUN[SYMBOLS[0]].tr_W
for s in SYMBOLS:
    for ev in EV[s].itertuples():
        for m, store in [(MODELS[1], XE), (MODELS[2], TE)]:
            b = peak_bar(s, m, ev)
            if b is None or b < PRE: excl[m] += 1; continue
            if not (SAV[(m, s)][b] > THR[(m, s)][b]): excl[m] += 1; continue
            store.append((s, ev.family, ev.amp, b))
say(f'model-active events: XGBoost {len(XE)} (excluded {excl[MODELS[1]]}), Transformer {len(TE)} (excluded {excl[MODELS[2]]}) of {sum(len(v) for v in EV.values())}; Transformer window {W_} bars')

model-active events: XGBoost 431 (excluded 169), Transformer 412 (excluded 188) of 600; Transformer window 16 bars


## 4 · Background and Shapley routine

In [5]:

# ---------------- new in E4: background drawn from CAL-3 (written before any TEST event is explained)
KX, KT, GAP_BG = 50, 20, 130
GROUPS3 = {'price': ['log_returns', 'hl_spread_ratio'], 'flow': ['trade_flow_imbalance', 'rush_order_density'], 'size_volume': ['large_trade_count', 'volume_acceleration']}
GN = list(GROUPS3); GI = [[FEATS.index(f_) for f_ in GROUPS3[g_]] for g_ in GN]; assert sorted(sum(GI, [])) == list(range(6))
CALN = 31 * 1440; FRC = {}; EVC = {}; ZC = {}; BGBARS = {}
for s in SYMBOLS:
    f = pd.read_parquet(L3/f'l3v3_{s}_CAL.parquet'); FRC[s] = f[f.replica == 0].reset_index(drop=True).iloc[:CALN][FEATS]
    e = pd.read_csv(L3/f'l3v3_{s}_CAL_events.csv'); EVC[s] = e[e.replica == 0].reset_index(drop=True)
    for m in MODELS[1:]: ZC[(m, s)] = np.asarray(B.normalise(FRC[s], MODES[m], N0[s]), 'float64')
    near = np.zeros(CALN, bool)
    for ev in EVC[s].itertuples(): near[max(0, ev.t0 - GAP_BG):min(CALN, ev.t0 + ev.dur + GAP_BG)] = True
    ok = ~near; ok[:PRE + 130] = False; ok &= np.isfinite(ZC[(MODELS[1], s)]).all(axis=1) & np.isfinite(ZC[(MODELS[2], s)]).all(axis=1)
    BGBARS[s] = np.sort(np.random.default_rng(11).choice(np.where(ok)[0], KX, replace=False))
    assert all(not near[b_] for b_ in BGBARS[s])
# 53-feature rows (XGBoost) and windows (Transformer) of the background bars
BGX = {s: np.vstack([xgb_x(s, ZC[(MODELS[1], s)], b_) for b_ in BGBARS[s]]) for s in SYMBOLS}
W_ = BUN[SYMBOLS[0]].tr_W; BGW = {s: np.stack([ZC[(MODELS[2], s)][b_ - W_ + 1:b_ + 1] for b_ in BGBARS[s][:KT]]) for s in SYMBOLS}
np.savez_compressed(OUT/'explain_background_v1.npz', **{f'{s}_xgb_rows': BGX[s] for s in SYMBOLS}, **{f'{s}_tr_windows': BGW[s] for s in SYMBOLS}, **{f'{s}_bars': BGBARS[s] for s in SYMBOLS})
say('background written before any TEST event is explained: ' + str({s: (BGX[s].shape, BGW[s].shape) for s in SYMBOLS}) + '; sha16 ' + sha_file(OUT/'explain_background_v1.npz'))

# ---------------- exact interventional (background-marginalised) Shapley values for the six features
NF = 6; MASKS = np.arange(2 ** NF); KEEP = ((MASKS[:, None] >> np.arange(NF)) & 1).astype(bool)          # KEEP[mask, j] = feature j kept at the event's value
POP = KEEP.sum(axis=1); from math import factorial
WSH = np.array([factorial(k) * factorial(NF - k - 1) / factorial(NF) for k in range(NF)])
def coalition_rows(x53, bgx):
    """rows (64 * K, 53): for every coalition, the event row with the derived columns of every non-kept feature taken from each background row"""
    K = len(bgx); R = np.repeat(x53[None, :], len(MASKS) * K, axis=0).reshape(len(MASKS), K, -1).copy()
    for j in range(NF):
        cols = GIDX[FEATS[j]]; drop = ~KEEP[:, j]
        R[np.ix_(drop, np.arange(K), cols)] = bgx[None, :, cols]
    return R.reshape(len(MASKS) * K, -1)
def shapley_from_v(v):
    """v[mask] coalition values (64,) -> exact Shapley values (6,)"""
    phi = np.zeros(NF)
    for j in range(NF):
        for mk in MASKS[~KEEP[:, j]]:
            phi[j] += WSH[POP[mk]] * (v[mk | (1 << j)] - v[mk])
    return phi
def xgb_margin_rows(s, X):
    dm = xgb.DMatrix(X, feature_names=c.FA_NAMES); return np.mean([b_.predict(dm, output_margin=True, iteration_range=(0, bi + 1)) for b_, bi in BUN[s].xgb], axis=0)
def xgb_v(s, x53, bgx):
    y = xgb_margin_rows(s, coalition_rows(x53, bgx)).reshape(len(MASKS), len(bgx)); return y.mean(axis=1)
FULL = 2 ** NF - 1
def drops_from_v(v):
    d6 = np.array([v[FULL] - v[FULL & ~(1 << j)] for j in range(NF)]); dg = np.array([v[FULL] - v[FULL & ~sum(1 << j for j in ix)] for ix in GI]); return d6, dg
def grp(v6): return np.array([float(np.sum(np.asarray(v6)[ix])) for ix in GI])
def tr_whatif(s, win, bgw):
    """drop of the Transformer output when the last DELLEN bars of feature j are replaced by those of each background window (mean over windows)"""
    o0 = tr_out(s, win); out = np.zeros(NF)
    for j in range(NF):
        ys = []
        for w_b in bgw:
            w2 = np.asarray(win, 'float32').copy(); w2[-DELLEN:, j] = np.asarray(w_b, 'float32')[-DELLEN:, j]; ys.append(tr_out(s, w2))
        out[j] = o0 - float(np.mean(ys))
    return out


background written before any TEST event is explained: {'BTCUSDT': ((50, 53), (20, 16, 6)), 'ETHUSDT': ((50, 53), (20, 16, 6))}; sha16 06dae715477d97c6


## 5 · Unit tests

In [6]:
s0 = SYMBOLS[0]; s_, f_, a_, b_ = XE[0] if XE else (s0, 1, 0, 2000); zx = Z[(MODELS[1], s_)]; X = xgb_x(s_, zx, b_); con, bias = xgb_shap(s_, X)
assert abs(con.sum() + bias - xgb_margin(s_, X)) <= 1e-3; say('PASS T01 SHAP additivity: contributions + bias equal the booster margin (first event)')
errs = []
for (s_, f_, a_, b_) in XE[:20]: errs.append(abs(xgb_margin(s_, xgb_x(s_, Z[(MODELS[1], s_)], b_)) - SAV[(MODELS[1], s_)][b_]))
assert max(errs) <= 1e-3, max(errs); say(f'PASS T02 XGBoost margin recomputed from a short slice equals the saved score on {len(errs)} event peak bars (max difference {max(errs):.1e})')
errs = []
for (s_, f_, a_, b_) in TE[:20]: errs.append(abs(tr_out(s_, Z[(MODELS[2], s_)][b_ - W_ + 1:b_ + 1]) - SAV[(MODELS[2], s_)][b_]))
assert max(errs) <= 5e-3, max(errs); say(f'PASS T03 Transformer output recomputed here equals the saved score on {len(errs)} event peak bars (max difference {max(errs):.1e}; tolerance 5e-3 because the saved scores come from another device, protocol amendment 1)')
rg = np.random.default_rng(0); rel = []
for (s_, f_, a_, b_) in TE[:20]:
    win = Z[(MODELS[2], s_)][b_ - W_ + 1:b_ + 1]; ai, g = tr_gradxinput(s_, win); d = rg.normal(size=win.shape) * 1e-2; lin = float((g * d).sum()); act = tr_out(s_, win + d) - tr_out(s_, win)
    if abs(lin) > 1e-6: rel.append(abs(act - lin) / abs(lin))
assert len(rel) and np.median(rel) <= 0.05, np.median(rel); say(f'PASS T04 gradient first-order check: median relative error {np.median(rel):.3f} over {len(rel)} windows')
win = Z[(MODELS[2], s_)][b_ - W_ + 1:b_ + 1] if TE else np.zeros((W_, 6)); w2 = delete_tr(win, 2); assert np.allclose(np.delete(w2, 2, axis=1), np.delete(win.astype('float32'), 2, axis=1)) and (w2[-DELLEN:, 2] == 0).all(); say('PASS T05 deleting feature j changes only column j (and only the last 64 bars)')
tg = explain_tgnn(); assert tg['available'] is False and tg['feature_attribution'] is None and tg['time_attribution'] is None; say('PASS T06 the TGNN explanation interface returns available = false and no numbers')
# T07 coalition rows: the full coalition is the event row; the empty coalition is the background; a single dropped feature changes only that feature's columns
x_ = np.arange(53, dtype='float64'); b_ = -np.arange(53, dtype='float64')[None, :] - 1; R_ = coalition_rows(x_, b_)
fc_ = sorted(sum([GIDX[f_] for f_ in FEATS], [])); xc_ = GIDX['x']; assert np.array_equal(R_[FULL], x_) and np.array_equal(R_[0][fc_], b_[0][fc_]) and np.array_equal(R_[0][xc_], x_[xc_]); j_ = 2; r_ = R_[FULL & ~(1 << j_)]; ch = np.where(r_ != x_)[0]; assert sorted(ch.tolist()) == sorted(GIDX[FEATS[j_]]); say('PASS T07 coalition rows: full = event, empty = background, one dropped feature changes only its own columns; cross-feature columns stay')
# T08 Shapley routine on a linear game reproduces the closed form and efficiency
w6 = np.array([1.0, -2.0, 0.5, 3.0, 0.0, 1.5]); xs = np.array([2.0, 1.0, -1.0, 0.5, 4.0, 0.0]); bs = np.array([0.5, -0.5, 0.2, -1.0, 1.0, 0.3]); v_ = np.array([sum(w6[j] * (xs[j] if KEEP[mk, j] else bs[j]) for j in range(NF)) for mk in MASKS])
ph_ = shapley_from_v(v_); assert np.allclose(ph_, w6 * (xs - bs), atol=1e-9) and abs(ph_.sum() - (v_[FULL] - v_[0])) < 1e-9; say('PASS T08 Shapley routine equals the closed form of a linear game and is efficient')
# T09 efficiency and full-coalition value on a real event, and the background is away from every CAL-3 event
if XE:
    s_, f_, a_, b_e = XE[0]; x53 = xgb_x(s_, Z[(MODELS[1], s_)], b_e)[0]; vv = xgb_v(s_, x53, BGX[s_]); ph = shapley_from_v(vv)
    assert abs(ph.sum() - (vv[FULL] - vv[0])) < 1e-6 and abs(vv[FULL] - xgb_margin(s_, x53[None, :])) <= 1e-3; say('PASS T09 efficiency (sum phi = v(all) - v(empty)) and v(all) equals the booster margin of the event row')
for s_ in SYMBOLS:
    for ev in EVC[s_].itertuples(): assert not ((BGBARS[s_] >= ev.t0 - GAP_BG) & (BGBARS[s_] < ev.t0 + ev.dur + GAP_BG)).any()
say('PASS T10 no background bar lies within 130 bars of a CAL-3 event')


PASS T01 SHAP additivity: contributions + bias equal the booster margin (first event)
PASS T02 XGBoost margin recomputed from a short slice equals the saved score on 20 event peak bars (max difference 0.0e+00)
PASS T03 Transformer output recomputed here equals the saved score on 20 event peak bars (max difference 6.4e-07; tolerance 5e-3 because the saved scores come from another device, protocol amendment 1)
PASS T04 gradient first-order check: median relative error 0.019 over 20 windows
PASS T05 deleting feature j changes only column j (and only the last 64 bars)
PASS T06 the TGNN explanation interface returns available = false and no numbers
PASS T07 coalition rows: full = event, empty = background, one dropped feature changes only its own columns; cross-feature columns stay
PASS T08 Shapley routine equals the closed form of a linear game and is efficient
PASS T09 efficiency (sum phi = v(all) - v(empty)) and v(all) equals the booster margin of the event row
PASS T10 no background bar

## 6 · Results (protocol E4 v1)

In [7]:

def spear(a, b):
    if np.std(a) == 0 or np.std(b) == 0: return 0.0
    return float(spearmanr(a, b)[0])
rg = np.random.default_rng(7); rows = []; EXACT = []
for (s, fam, amp, b) in XE:
    z = Z[(MODELS[1], s)]; X = xgb_x(s, z, b); x53 = X[0]; vv = xgb_v(s, x53, BGX[s]); phi = shapley_from_v(vv); d6, dg = drops_from_v(vv); con, bias = xgb_shap(s, X); EXACT.append(abs(phi.sum() - (vv[FULL] - vv[0])))
    rows.append({'model': MODELS[1], 'sym': s, 'family': int(fam), 'attr': phi, 'D': d6, 'DG': dg, 'tree': shap6(con), 'rand': rg.permutation(6).astype(float), 'randg': rg.permutation(3).astype(float)})
for (s, fam, amp, b) in TE:
    z = Z[(MODELS[2], s)]; win = z[b - W_ + 1:b + 1]; ai, g = tr_gradxinput(s, win); at6 = np.abs(ai).sum(axis=0); wi = tr_whatif(s, win, BGW[s])
    rows.append({'model': MODELS[2], 'sym': s, 'family': int(fam), 'attr': at6, 'D': wi, 'DG': np.array([wi[ix].sum() for ix in GI]), 'tree': at6, 'rand': rg.permutation(6).astype(float), 'randg': rg.permutation(3).astype(float)})
R = pd.DataFrame(rows)
def boot(x, n=NBOOT, seed=0):
    x = np.asarray(x, float); r_ = np.random.default_rng(seed); return np.percentile([x[r_.integers(0, len(x), len(x))].mean() for _ in range(n)], [2.5, 97.5])
def boot_diff(a, b_, n=NBOOT, seed=0):
    d = np.asarray(a, float) - np.asarray(b_, float); r_ = np.random.default_rng(seed); return np.percentile([d[r_.integers(0, len(d), len(d))].mean() for _ in range(n)], [2.5, 97.5])
OUTR = {}; tab = []
for m in [MODELS[1], MODELS[2]]:
    r = R[R.model == m]
    if len(r) < 10: OUTR[m] = None; continue
    sg = [SIGN[int(f)] for f in r.family]; chance_sig = np.array([len(x) / 6 for x in sg])
    att = np.array([float(FEATS[int(np.argmax(a))] in x) for a, x in zip(r.attr, sg)]); occ = np.array([float(FEATS[int(np.argmax(d))] in x) for d, x in zip(r.D, sg)])
    gh = np.array([float(np.argmax(grp(a)) == np.argmax(d)) for a, d in zip(r.attr, r.DG)]); ghr = np.array([float(np.argmax(a) == np.argmax(d)) for a, d in zip(r.randg, r.DG)])
    top2 = np.array([float(np.argmax(a) in np.argsort(-d)[:2]) for a, d in zip(r.attr, r.D)]); top2r = np.array([float(np.argmax(a) in np.argsort(-d)[:2]) for a, d in zip(r.rand, r.D)])
    rho = np.array([spear(a, d) for a, d in zip(r.attr, r.D)]); ghT = np.array([float(np.argmax(grp(a)) == np.argmax(d)) for a, d in zip(r.tree, r.DG)])
    cg, c2, ca, cd, cx = boot(gh), boot(top2), boot(att - chance_sig), boot_diff(occ, att), boot(occ - chance_sig)
    OUTR[m] = {'n': len(r), 'group_hit': float(gh.mean()), 'group_ci': [float(x) for x in cg], 'group_random': float(ghr.mean()), 'top2': float(top2.mean()), 'top2_ci': [float(x) for x in c2], 'top2_random': float(top2r.mean()), 'spearman': float(rho.mean()),
               'attr_sig': float(att.mean()), 'sig_chance': float(chance_sig.mean()), 'attr_minus_chance_ci': [float(x) for x in ca], 'whatif_sig': float(occ.mean()), 'whatif_minus_attr': float((occ - att).mean()), 'whatif_minus_attr_ci': [float(x) for x in cd], 'whatif_minus_chance_ci': [float(x) for x in cx],
               'treeshap_group_hit_vs_same_drops': float(ghT.mean()),
               'H_E1': bool(gh.mean() >= 0.60 and cg[0] > 1 / 3), 'H_E2': bool(top2.mean() >= 0.65 and c2[0] > 1 / 3), 'H_E3': bool(att.mean() >= chance_sig.mean() + 0.15 and ca[0] > 0), 'H_E4': bool(occ.mean() >= chance_sig.mean() + 0.15 and cd[0] > -0.03)}
    tab.append({'model': m, 'active TEST events': len(r), 'group hit': gh.mean(), 'group random': ghr.mean(), 'group CI low': cg[0], 'top-2 agreement': top2.mean(), 'top-2 random': top2r.mean(), 'top-2 CI low': c2[0], 'Spearman': rho.mean(),
                'attribution top-1 in signature': att.mean(), 'what-if top-1 in signature': occ.mean(), 'signature chance': chance_sig.mean(), 'path-dependent TreeSHAP (or gradient) group hit vs same drops': ghT.mean()})
TB = pd.DataFrame(tab); TB.to_csv(OUT/'e4_faithfulness_table.csv', index=False); say('--- explanations on model-active TEST events (Sep and Oct); XGBoost = interventional Shapley, background K=50; Transformer what-if with 20 background windows ---'); say(TB.round(3).T.to_string())
fr = []
for m in [MODELS[1], MODELS[2]]:
    r = R[R.model == m]
    for f in sorted(r.family.unique()):
        q = r[r.family == f]; sg = SIGN[f]; fr.append({'model': m, 'family': B.FAMS[int(f)], 'events': len(q), 'group hit': float(np.mean([np.argmax(grp(a)) == np.argmax(d) for a, d in zip(q.attr, q.DG)])), 'top-2 agreement': float(np.mean([np.argmax(a) in np.argsort(-d)[:2] for a, d in zip(q.attr, q.D)])),
                   'what-if top-1 in signature': float(np.mean([FEATS[int(np.argmax(d))] in sg for d in q.D])), 'attribution top-1 in signature': float(np.mean([FEATS[int(np.argmax(a))] in sg for a in q.attr])), 'signature chance': len(sg) / 6})
FF = pd.DataFrame(fr); FF.to_csv(OUT/'e4_family_table.csv', index=False); say(''); say('--- per family ---'); say(FF.round(2).to_string(index=False))
say(''); say('=== pre-registered hypotheses (protocol E4 v1; saved TEST scores, recomputed explanations) ===')
HYP = {}; ox, ot = OUTR.get(MODELS[1]), OUTR.get(MODELS[2])
if ox is None: say('XGBoost: fewer than 10 model-active events -> H-E1y..3y FAIL'); HYP.update({'H-E1y': False, 'H-E2y': False, 'H-E3y': False})
else:
    say(f'H-E1y XGBoost interventional Shapley: group hit >= 0.60 and lower bound > 1/3: {ox["group_hit"]:.3f} ({ox["group_ci"][0]:.3f}..{ox["group_ci"][1]:.3f}), random {ox["group_random"]:.3f} (path-dependent TreeSHAP on the same drops: {ox["treeshap_group_hit_vs_same_drops"]:.3f}) -> {"HOLDS" if ox["H_E1"] else "FAILS"}')
    say(f'H-E2y XGBoost interventional Shapley: top-1 within the two largest drops >= 0.65 and lower bound > 1/3: {ox["top2"]:.3f} ({ox["top2_ci"][0]:.3f}..{ox["top2_ci"][1]:.3f}), random {ox["top2_random"]:.3f} -> {"HOLDS" if ox["H_E2"] else "FAILS"}')
    say(f'H-E3y XGBoost interventional Shapley: top-1 in the injected signature >= chance + 0.15 and lower bound of (hit - chance) > 0: {ox["attr_sig"]:.3f} (chance {ox["sig_chance"]:.3f}, lower bound of the difference {ox["attr_minus_chance_ci"][0]:+.3f}) -> {"HOLDS" if ox["H_E3"] else "FAILS"}')
    HYP.update({'H-E1y': ox['H_E1'], 'H-E2y': ox['H_E2'], 'H-E3y': ox['H_E3']})
if ot is None: say('Transformer: fewer than 10 model-active events -> H-E4y FAILS'); HYP['H-E4y'] = False
else:
    say(f'H-E4y Transformer background what-if: top-1 in the injected signature >= chance + 0.15 and not worse than gradient x input by > 0.03: {ot["whatif_sig"]:.3f} (chance {ot["sig_chance"]:.3f}, attribution {ot["attr_sig"]:.3f}, difference {ot["whatif_minus_attr"]:+.3f} [{ot["whatif_minus_attr_ci"][0]:+.3f}..{ot["whatif_minus_attr_ci"][1]:+.3f}]) -> {"HOLDS" if ot["H_E4"] else "FAILS"}')
    HYP['H-E4y'] = ot['H_E4']
HYP['H-E5y'] = bool(len(EXACT) == 0 or max(EXACT) <= 1e-6); say(f'H-E5y exactness: efficiency of the Shapley values (max error {max(EXACT) if EXACT else 0:.1e}), full-coalition value, toy linear game, background away from CAL-3 events (T07-T10) -> {"HOLDS" if HYP["H-E5y"] else "FAILS"}')
say(f'{sum(HYP.values())} of {len(HYP)} hold. Failed hypotheses are reported as failed; nothing was retuned. Explanations describe the model score, not the cause of a market event.')
PRODUCT = {'xgboost_group_reason_supported': bool(HYP.get('H-E1y') and HYP.get('H-E3y')), 'transformer_whatif_supported': bool(HYP.get('H-E4y'))}
DECL = {'status': 'DECLARED: product rules of protocol E4 v1 applied to the results of notebook 59', 'protocol_e4_v1_sha256': PHE, 'hypotheses': HYP, 'results': OUTR, 'product': PRODUCT, 'groups': GROUPS3, 'background_file': 'explain_background_v1.npz', 'background_sha16': sha_file(OUT/'explain_background_v1.npz'), 'K': {'xgboost': KX, 'transformer': KT}, 'mock': MOCK}
(OUT/'declared_explain_v4.json').write_text(json.dumps(DECL, indent=1, default=str)); (OUT/'e4_share_block.txt').write_text('\n'.join(LINES))
say('Saved: declared_explain_v4.json, explain_background_v1.npz, e4_faithfulness_table.csv, e4_family_table.csv, e4_share_block.txt in models_b2_v1/ . Send the share block to Claude.')


--- explanations on model-active TEST events (Sep and Oct); XGBoost = interventional Shapley, background K=50; Transformer what-if with 20 background windows ---
                                                                        0               1
model                                                          XGBoost v2  Transformer v2
active TEST events                                                    431             412
group hit                                                           0.977           0.617
group random                                                        0.353           0.342
group CI low                                                        0.961           0.568
top-2 agreement                                                     0.988           0.541
top-2 random                                                         0.35           0.337
top-2 CI low                                                        0.977           0.495
Spearman                    